# Configuring frequency units

A frequency's input unit tells scikit-rf what the supplied numbers mean.
Its display unit controls how those frequencies are represented. Internally,
`Frequency.f` always stores values in Hz; `Frequency.f_scaled` expresses the
same frequencies in `Frequency.unit`.

Set `rf.constants.FREQ_UNIT_DEFAULT` to choose a common unit for newly created
frequency objects and networks, including networks read from Touchstone files.
This setting applies to the current Python process and needs no configuration
file. The examples below save the current setting and restore it at the end.

In [ ]:
from io import StringIO

import numpy as np

import skrf as rf

original_unit = rf.constants.FREQ_UNIT_DEFAULT

## Keeping input units

The factory setting is `None`. It preserves explicit constructor units and
file units. When `unit` is omitted from `Frequency` or `Frequency.from_f`,
the input is interpreted in Hz.

In [ ]:
rf.constants.FREQ_UNIT_DEFAULT = None

omitted = rf.Frequency(1, 2, 2)
original = rf.Frequency(1000, 2000, 2, unit="MHz")

print("Omitted unit:", omitted.unit, omitted.f_scaled)
print("Explicit unit:", original.unit, original.f_scaled)
assert omitted.unit == "Hz"
assert original.unit == "MHz"
np.testing.assert_allclose(omitted.f, [1, 2])
np.testing.assert_allclose(original.f, [1e9, 2e9])

## Choosing a common unit

With a global unit of GHz, omitted Frequency input units are interpreted as
GHz. An explicit input unit still determines the meaning of the input values,
but the resulting object uses GHz for its representation.

All three constructions below describe the same 1-2 GHz band. For each object,
`f` is `[1e9, 2e9]`, `unit` is `"GHz"`, and `f_scaled` is `[1, 2]`.

In [ ]:
rf.constants.FREQ_UNIT_DEFAULT = "GHz"

implicit = rf.Frequency(1, 2, 2)
explicit = rf.Frequency(1000, 2000, 2, unit="MHz")
from_values = rf.Frequency.from_f([1000, 2000], unit="MHz")

for frequency in (implicit, explicit, from_values):
    print(frequency.f, "Hz ->", frequency.f_scaled, frequency.unit)
    assert frequency.unit == "GHz"
    np.testing.assert_allclose(frequency.f, [1e9, 2e9])
    np.testing.assert_allclose(frequency.f_scaled, [1, 2])

## Reading Touchstone data

A Touchstone header defines the unit of the file's frequency values. scikit-rf
uses that unit to read the data before applying the configured global unit.
The absolute frequencies, scattering parameters and impedances are unchanged.
For files containing noise data, the noise frequency axis also uses the
configured unit.

This small Touchstone example contains two frequencies, 1000 and 2000 MHz.
We use an in-memory file so the notebook needs no downloads or data files.
For a file on disk, use `rf.Network("example.s1p")` in the same way.

In [ ]:
touchstone_data = "# MHz S RI R 50\n1000 0.1 0\n2000 0.2 0\n"

with StringIO(touchstone_data) as stream:
    stream.name = "example.s1p"
    rf.constants.FREQ_UNIT_DEFAULT = None
    file_units = rf.Network(stream)

with StringIO(touchstone_data) as stream:
    stream.name = "example.s1p"
    rf.constants.FREQ_UNIT_DEFAULT = "GHz"
    configured = rf.Network(stream)

print("File units:", file_units.frequency.f_scaled, file_units.frequency.unit)
print("Configured units:", configured.frequency.f_scaled, configured.frequency.unit)
assert file_units.frequency.unit == "MHz"
assert configured.frequency.unit == "GHz"
np.testing.assert_allclose(configured.f, [1e9, 2e9])
np.testing.assert_allclose(configured.frequency.f_scaled, [1, 2])
np.testing.assert_array_equal(configured.f, file_units.f)
np.testing.assert_array_equal(configured.s, file_units.s)
np.testing.assert_array_equal(configured.z0, file_units.z0)

## Supplying raw frequencies to a Network

`Network(f=...)` keeps its existing input convention: raw numbers are in Hz
unless `f_unit` specifies another input unit. This differs from omitting
`unit` in a Frequency constructor, which uses the configured unit for input.
The resulting Network frequency unit follows the global setting in both cases.

In [ ]:
raw_hz = rf.Network(f=[1e9, 2e9], s=[0.1, 0.2])
raw_mhz = rf.Network(f=[1000, 2000], f_unit="MHz", s=[0.1, 0.2])

for network in (raw_hz, raw_mhz):
    print(network.frequency.f_scaled, network.frequency.unit)
    assert network.frequency.unit == "GHz"
    np.testing.assert_allclose(network.f, [1e9, 2e9])

## Existing objects and copies

Changing the global setting does not change existing objects. The `original`
frequency created earlier still uses MHz, and its copies and slices retain
that unit. You can explicitly assign `.unit` to change an individual object's
representation without changing its absolute frequencies.

Constructing a new Network from an existing Frequency applies the current
global setting to the network's copy and leaves the source unchanged.

In [ ]:
copied = original.copy()
sliced = original[1:]
network = rf.Network(frequency=original, s=[0.1, 0.2])

assert original.unit == copied.unit == sliced.unit == "MHz"
assert network.frequency.unit == "GHz"
np.testing.assert_array_equal(network.f, original.f)

copied.unit = "kHz"
print("Original:", original.f_scaled, original.unit)
print("Modified copy:", copied.f_scaled, copied.unit)
print("New network:", network.frequency.f_scaled, network.frequency.unit)
assert original.unit == "MHz"
np.testing.assert_array_equal(copied.f, original.f)

## Restoring the previous setting

Restore the setting saved at the start so that later work in this kernel uses
the same configuration as before the tutorial. The objects already created
above keep their units.

In [ ]:
rf.constants.FREQ_UNIT_DEFAULT = original_unit